In [ ]:
from google.colab import files
print("Upload chunks.parquet")
files.upload()  # upload data/chunks.parquet

print("Upload models/finbert_tone/ (zip)")
files.upload()  # upload finbert_tone.zip

Upload chunks.parquet


Saving chunks.parquet to chunks.parquet
Upload models/finbert_tone/ (zip)


Saving finbert_tone.zip to finbert_tone.zip


In [ ]:
#Upload finbert_tone.zip on drive for faster upload and put it in your file that you need
from google.colab import drive
drive.mount('/content/drive')


ValueError: mount failed

In [ ]:
import zipfile, os
zipfile.ZipFile("finbert_tone.zip").extractall("models/finbert_tone")
os.makedirs("data", exist_ok=True)
os.rename("chunks.parquet", "data/chunks.parquet")
!pip install -q transformers pyarrow

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda")
print(torch.cuda.get_device_name(0))

df = pd.read_parquet("data/chunks.parquet")
df = df[df["section"] == "Prepared"].copy()
print(f"Scoring {len(df)} chunks")

tokenizer = AutoTokenizer.from_pretrained("models/finbert_tone")
model = AutoModelForSequenceClassification.from_pretrained("models/finbert_tone")
model.to(device)
model.eval()

BATCH_SIZE = 64
texts = df["content"].tolist()
all_probs = []

for i in range(0, len(texts), BATCH_SIZE):
    batch = tokenizer(texts[i:i+BATCH_SIZE], truncation=True,
                      padding=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        logits = model(input_ids=batch["input_ids"].to(device),
                       attention_mask=batch["attention_mask"].to(device)).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    all_probs.append(probs)
    if (i // BATCH_SIZE + 1) % 200 == 0:
        print(f"{i+BATCH_SIZE}/{len(texts)}")

probs_arr = np.vstack(all_probs)
df["p_optimistic"] = probs_arr[:, 0]
df["p_cautious"]   = probs_arr[:, 1]
df["p_negative"]   = probs_arr[:, 2]

tone_scores = df.groupby(["ticker","sector","company","date","quarter"]).agg(
    optimistic_share=("p_optimistic","mean"),
    cautious_share  =("p_cautious",  "mean"),
    negative_share  =("p_negative",  "mean"),
    n_chunks        =("chunk_id",    "count"),
).reset_index()

tone_scores.to_parquet("tone_scores.parquet", index=False)
print(f"Done — {len(tone_scores)} calls")
print(tone_scores[["cautious_share","optimistic_share","negative_share"]].describe().round(4))

Tesla T4
Scoring 183491 chunks


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

12800/183491
25600/183491
38400/183491
51200/183491
64000/183491
76800/183491
89600/183491
102400/183491
115200/183491
128000/183491
140800/183491
153600/183491
166400/183491
179200/183491
Done — 15039 calls
       cautious_share  optimistic_share  negative_share
count      15039.0000        15039.0000      15039.0000
mean           0.3991            0.5834          0.0175
std            0.1411            0.1451          0.0070
min            0.0424            0.0132          0.0087
25%            0.2943            0.4922          0.0142
50%            0.3829            0.6005          0.0162
75%            0.4883            0.6912          0.0189
max            0.9704            0.9489          0.2351


In [ ]:
from google.colab import files
files.download("tone_scores.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>